# Imports

In [21]:
import torch
import sys


from torch_geometric.datasets import Planetoid

c:\faculdade\Tabalho-Final-XAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:
import random
import torch.nn.functional as F
from torch_geometric.nn import  HANConv
from torch_geometric.datasets import DBLP
import os
import pandas as pd

In [23]:
from itertools import combinations
from tqdm import tqdm
import re
from scipy.stats import spearmanr

In [24]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

In [25]:
import numpy as np
import warnings
import pickle
from pathlib import Path

In [26]:
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import PGExplainer
from torch_geometric.explain.config import ModelConfig

In [ ]:
from torch_geometric.data import HeteroData
from captum.attr import Saliency
from captum.attr import IntegratedGradients
import torch

# Dados

In [7]:
def load_dataset(name):

    if name in [
        "Cora",
        "CiteSeer",
        "PubMed"
    ]:

        dataset = Planetoid(
            root=f"data/{name}",
            name=name
        )

        return dataset

    elif name == "DBLP":

        dataset = DBLP(
            root="data/DBLP"
        )

        return dataset

    else:

        raise ValueError(
            f"Dataset {name} não suportado."
        )

DBLP

In [9]:
DBdataset = DBLP(
    root='data/DBLP'
)

In [10]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

In [11]:
dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

In [12]:
print(dblp_new.metadata())

(['author', 'paper', 'term'], [('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')])


In [13]:
print(dblp_new.node_types)

['author', 'paper', 'term']


In [14]:
print(dblp_new.edge_types)

[('author', 'to', 'paper'), ('paper', 'to', 'author'), ('paper', 'to', 'term'), ('term', 'to', 'paper')]


In [15]:
print(dblp_new["author"])

{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}


In [16]:
for node_type in dblp_new.node_types:

    print()
    print(node_type)
    print(dblp_new[node_type])


author
{'x': tensor([[0., 0., 1.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]]), 'y': tensor([2, 2, 3,  ..., 0, 0, 0]), 'train_mask': tensor([False, False, False,  ..., False, False, False]), 'val_mask': tensor([False, False,  True,  ..., False, False, False]), 'test_mask': tensor([ True,  True, False,  ...,  True,  True,  True])}

paper
{'x': tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])}

term
{'x': tensor([[-0.6924, -0.4659,  1.1540,  ...,  0.9178,  0.1995, -0.6360],
        [ 1.2031, -0.4003,  0.0740,  ...,  1.3262, -0.3325,  0.8198],
        [ 0.3748,  0.5731,  0.4802,  ...,  1.1522,  0.6010,

In [17]:
for node_type in dblp_new.node_types:

    print(node_type)
    print("num_nodes =", dblp_new[node_type].num_nodes)

    if 'x' in dblp_new[node_type]:
        print("x.shape =", dblp_new[node_type].x.shape)

    print()

author
num_nodes = 4057
x.shape = torch.Size([4057, 334])

paper
num_nodes = 14328
x.shape = torch.Size([14328, 4231])

term
num_nodes = 7723
x.shape = torch.Size([7723, 50])



# Modelo

In [68]:
class HAN(torch.nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        metadata,
        heads=8,
        dropout=0.5
    ):
        super().__init__()

        self.conv = HANConv(
            in_channels=in_channels,
            out_channels=hidden_channels,
            metadata=metadata,
            heads=heads
        )

        self.dropout = torch.nn.Dropout(
            p=dropout
        )

        self.lin = torch.nn.Linear(
            hidden_channels,
            out_channels
        )

    def forward(
        self,
        x_dict,
        edge_index_dict,
        return_embeddings=False
    ):

        x_dict = self.conv(
            x_dict,
            edge_index_dict
        )

        x_author = x_dict["author"]

        x_author = self.dropout(
            x_author
        )

        out = self.lin(
            x_author
        )

        if return_embeddings:
            return out, x_author

        return out

# Treinamento

## Apoio

In [8]:
def train(model,data,optimizer):
    model.train()
    optimizer.zero_grad()

    out = model(data.x_dict, data.edge_index_dict)

    criterion = torch.nn.CrossEntropyLoss()

    loss = criterion(out[data['author'].train_mask],
                      data['author'].y[data['author'].train_mask])

    loss.backward()
    optimizer.step()

    return loss.item()

In [9]:
@torch.no_grad()
def evaluate(model, data):

    model.eval()

    out = model(
        data.x_dict,
        data.edge_index_dict
    )

    pred = out.argmax(dim=1)

    train_acc = (
        pred[data['author'].train_mask]
        ==
        data['author'].y[data['author'].train_mask]
    ).float().mean()

    val_acc = (
        pred[data['author'].val_mask]
        ==
        data['author'].y[data['author'].val_mask]
    ).float().mean()

    test_acc = (
        pred[data['author'].test_mask]
        ==
        data['author'].y[data['author'].test_mask]
    ).float().mean()

    return (
        train_acc.item(),
        val_acc.item(),
        test_acc.item()
    )

NameError: name 'torch' is not defined

## Baseline

In [45]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

data = dblp.to(device)

model = HAN(
    in_channels=-1,  # PyG infere automaticamente
    hidden_channels=64,
    out_channels=4,  # DBLP tem 4 classes de autores
    metadata=metadata,
    heads=8
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=0.001)

In [ ]:
for epoch in range(1, 101):
    loss = train(model,data,optimizer)
    acc = evaluate(model,data)

    print(f"Epoch {epoch:03d}, Loss: {loss:.4f}, Acc: {acc:.4f}")

Epoch 001, Loss: 1.3948, Acc: 0.4105
Epoch 002, Loss: 1.3614, Acc: 0.5570
Epoch 003, Loss: 1.3223, Acc: 0.6033
Epoch 004, Loss: 1.2740, Acc: 0.6257
Epoch 005, Loss: 1.2194, Acc: 0.6371
Epoch 006, Loss: 1.1600, Acc: 0.6472
Epoch 007, Loss: 1.0962, Acc: 0.6641
Epoch 008, Loss: 1.0286, Acc: 0.6779
Epoch 009, Loss: 0.9583, Acc: 0.6905
Epoch 010, Loss: 0.8865, Acc: 0.7006
Epoch 011, Loss: 0.8143, Acc: 0.7080
Epoch 012, Loss: 0.7431, Acc: 0.7169
Epoch 013, Loss: 0.6740, Acc: 0.7277
Epoch 014, Loss: 0.6079, Acc: 0.7353
Epoch 015, Loss: 0.5455, Acc: 0.7452
Epoch 016, Loss: 0.4875, Acc: 0.7538
Epoch 017, Loss: 0.4341, Acc: 0.7651
Epoch 018, Loss: 0.3856, Acc: 0.7753
Epoch 019, Loss: 0.3419, Acc: 0.7826
Epoch 020, Loss: 0.3031, Acc: 0.7885
Epoch 021, Loss: 0.2690, Acc: 0.7967
Epoch 022, Loss: 0.2392, Acc: 0.8017
Epoch 023, Loss: 0.2135, Acc: 0.8038
Epoch 024, Loss: 0.1915, Acc: 0.8081
Epoch 025, Loss: 0.1727, Acc: 0.8084
Epoch 026, Loss: 0.1566, Acc: 0.8075
Epoch 027, Loss: 0.1429, Acc: 0.8066
E

## Variantes

In [25]:
def generate_han_variants():

    variants = []

    for hidden_channels in [32, 64, 128]:

        for heads in [2, 4, 8]:

            for dropout in [0.3, 0.5, 0.7]:

                variants.append({

                    "hidden_channels":
                        hidden_channels,

                    "heads":
                        heads,

                    "dropout":
                        dropout

                })

    return variants

In [48]:
SAVE_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

results = []

dataset_name = "DBLP"

print(f"\n{'='*50}")
print(f"Dataset: {dataset_name}")
print(f"{'='*50}")

data = dblp_new

variants = generate_han_variants()

for variant_id, config in enumerate(variants):

    print(
        f"\nVariant {variant_id}"
    )

    model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=config["hidden_channels"],

        out_channels=4,

        heads=config["heads"],

        dropout=config["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=0.005,

        weight_decay=5e-4

    )

    best_val = 0
    best_test = 0

    for epoch in range(1, 201):

        loss = train(
            model,
            data,
            optimizer
        )

        train_acc, val_acc, test_acc = evaluate(
            model,
            data
        )

        if val_acc > best_val:

            best_val = val_acc
            best_test = test_acc

    result = {

        "dataset": dataset_name,

        "model": "HAN",

        "variant_id": variant_id,

        "hidden_channels":
            config["hidden_channels"],

        "heads":
            config["heads"],

        "dropout":
            config["dropout"],

        "best_val":
            best_val,

        "best_test":
            best_test

    }

    results.append(result)

    print(
        f"Variant {variant_id} | "
        f"Val={best_val:.4f} | "
        f"Test={best_test:.4f}"
    )



Dataset: DBLP

Variant 0
Variant 0 | Val=0.7850 | Test=0.8014

Variant 1
Variant 1 | Val=0.7775 | Test=0.8041

Variant 2
Variant 2 | Val=0.7925 | Test=0.8078

Variant 3
Variant 3 | Val=0.7900 | Test=0.8075

Variant 4
Variant 4 | Val=0.7925 | Test=0.8124

Variant 5
Variant 5 | Val=0.7800 | Test=0.8103

Variant 6
Variant 6 | Val=0.7925 | Test=0.8020

Variant 7
Variant 7 | Val=0.7950 | Test=0.8096

Variant 8
Variant 8 | Val=0.7950 | Test=0.8099

Variant 9
Variant 9 | Val=0.7825 | Test=0.8035

Variant 10
Variant 10 | Val=0.7800 | Test=0.8121

Variant 11
Variant 11 | Val=0.7950 | Test=0.8142

Variant 12
Variant 12 | Val=0.7850 | Test=0.8010

Variant 13
Variant 13 | Val=0.7925 | Test=0.8133

Variant 14
Variant 14 | Val=0.7900 | Test=0.8115

Variant 15
Variant 15 | Val=0.7925 | Test=0.8075

Variant 16
Variant 16 | Val=0.7900 | Test=0.8112

Variant 17
Variant 17 | Val=0.7925 | Test=0.8201

Variant 18
Variant 18 | Val=0.7850 | Test=0.8118

Variant 19
Variant 19 | Val=0.7800 | Test=0.8112

Vari

In [53]:
results_df = pd.DataFrame(results)
results_df

,dataset,model,variant_id,hidden_channels,heads,dropout,best_val,best_test
0,DBLP,HAN,0,32,2,0.3,0.7850,0.801351
1,DBLP,HAN,1,32,2,0.5,0.7775,0.804114
2,DBLP,HAN,2,32,2,0.7,0.7925,0.807799
3,DBLP,HAN,3,32,4,0.3,0.7900,0.807492
4,DBLP,HAN,4,32,4,0.5,0.7925,0.812404
5,DBLP,HAN,5,32,4,0.7,0.7800,0.810255
6,DBLP,HAN,6,32,8,0.3,0.7925,0.801965
7,DBLP,HAN,7,32,8,0.5,0.7950,0.809641
8,DBLP,HAN,8,32,8,0.7,0.7950,0.809948
9,DBLP,HAN,9,64,2,0.3,0.7825,0.803500


In [50]:
top_models = (

    results_df

    .sort_values(

        by="best_test",

        ascending=False

    )

    .head(3)

)

In [52]:
for _, row in top_models.iterrows():

    filename = (

        f"DBLP_HAN"

        f"_h{row['hidden_channels']}"

        f"_heads{row['heads']}"

        f"_d{row['dropout']}"

        ".pth"

    )
    
    model = model = HAN(
        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=row["hidden_channels"],

        out_channels=4,

        heads=row["heads"],

        dropout=row["dropout"]

    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.01,
        weight_decay=5e-4
    )

    for epoch in range(200):

        train(
            model,
            data,
            optimizer
        )

    torch.save(

        model.state_dict(),

        os.path.join(
            SAVE_DIR,
            filename
        )

    )

    print(
        f"Modelo salvo: {filename}"
    )

Modelo salvo: DBLP_HAN_h64_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads8_d0.7.pth
Modelo salvo: DBLP_HAN_h128_heads2_d0.7.pth


# Conformidade

In [19]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

### Dados

In [20]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

### Apoio

In [18]:
from itertools import combinations
import pandas as pd
@torch.no_grad()
def pairwise_agreement(
    models,
    data,
    mask
):

    predictions = {}

    for model_name, model in models.items():

        model.eval()

        out = model(

            data.x_dict,

            data.edge_index_dict

        )

        predictions[
            model_name
        ] = out.argmax(dim=1).cpu()

    results = []

    model_ids = list(
        predictions.keys()
    )

    for model_a, model_b in combinations(
        model_ids,
        2
    ):

        pred_a = predictions[
            model_a
        ][mask.cpu()]

        pred_b = predictions[
            model_b
        ][mask.cpu()]

        agreement = (

            pred_a == pred_b

        ).float().mean().item()

        results.append({

            "model_a":
                model_a,

            "model_b":
                model_b,

            "agreement":
                agreement

        })

    return pd.DataFrame(
        results
    )

NameError: name 'torch' is not defined

In [ ]:
from itertools import combinations
import pandas as pd
@torch.no_grad()
def full_agreement_nodes(
    models,
    data,
    mask
):

    predictions = []

    for model in models.values():

        model.eval()

        out = model(

            data.x_dict,

            data.edge_index_dict

        )

        predictions.append(

            out.argmax(dim=1).cpu()

        )

    agreement_mask = torch.ones_like(

        predictions[0],

        dtype=torch.bool

    )

    for pred in predictions[1:]:

        agreement_mask &= (

            pred == predictions[0]

        )

    agreement_mask &= mask.cpu()

    return agreement_mask

In [ ]:
def load_han_model(
    hidden_channels,
    heads,
    dropout,
    data,
    device,
    save_dir
):

    model = HAN(

        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=hidden_channels,

        out_channels=4,

        heads=heads,

        dropout=dropout

    ).to(device)

    filename = (

        f"DBLP_HAN"

        f"_h{hidden_channels}"

        f"_heads{heads}"

        f"_d{dropout}.pth"

    )

    path = os.path.join(
        save_dir,
        filename
    )

    model.load_state_dict(

        torch.load(
            path,
            map_location=device
        )
    )

    model.eval()

    return model

In [ ]:
def load_han_models(
    save_dir,
    data,
    device
):

    models = {}

    for filename in os.listdir(save_dir):

        if not filename.endswith(".pth"):
            continue

        if not filename.startswith("DBLP_HAN"):
            continue

        parts = filename.replace(
            ".pth",
            ""
        ).split("_")

        hidden_channels = int(
            parts[2][1:]
        )

        heads = int(
            parts[3].replace(
                "heads",
                ""
            )
        )

        dropout = float(
            parts[4][1:]
        )

        model = HAN(

            in_channels={

                node_type:
                data[node_type].num_features

                for node_type in data.node_types

                if 'x' in data[node_type]

            },

            metadata=data.metadata(),

            hidden_channels=hidden_channels,

            out_channels=4,

            heads=heads,

            dropout=dropout

        ).to(device)

        model.load_state_dict(

            torch.load(

                os.path.join(
                    save_dir,
                    filename
                ),

                map_location=device

            )

        )

        model.eval()

        models[
            filename
        ] = model

    return models

### DBLP

HAN

In [26]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)

In [27]:
han_models.keys()

dict_keys(['DBLP_HAN_h128_heads2_d0.7.pth', 'DBLP_HAN_h128_heads8_d0.7.pth', 'DBLP_HAN_h64_heads8_d0.7.pth'])

In [32]:
agreement_df = pairwise_agreement(

    han_models,

    dblp_new,

    dblp_new["author"].test_mask

)

agreement_df

,model_a,model_b,agreement
0,DBLP_HAN_h128_heads2_d0.7.pth,DBLP_HAN_h128_heads8_d0.7.pth,0.903285
1,DBLP_HAN_h128_heads2_d0.7.pth,DBLP_HAN_h64_heads8_d0.7.pth,0.898373
2,DBLP_HAN_h128_heads8_d0.7.pth,DBLP_HAN_h64_heads8_d0.7.pth,0.957016


In [35]:
agreement_mask = full_agreement_nodes(

    han_models,

    dblp_new,

    dblp_new["author"].test_mask

)
agreement_rate = (

    agreement_mask.sum().item()

    /

    dblp_new["author"].test_mask.sum().item()

)

print(
    f"Concordância total: {agreement_rate:.2%}"
)

Concordância total: 88.09%


# Estabilidade

In [36]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

## Dados

In [37]:
dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

## Apoio

In [16]:


warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

warnings.filterwarnings(
    "ignore"
)

NameError: name 'warnings' is not defined

In [ ]:
def load_han_model(
    hidden_channels,
    heads,
    dropout,
    data,
    device,
    save_dir
):

    model = HAN(

        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=hidden_channels,

        out_channels=4,

        heads=heads,

        dropout=dropout

    ).to(device)

    filename = (

        f"DBLP_HAN"

        f"_h{hidden_channels}"

        f"_heads{heads}"

        f"_d{dropout}.pth"

    )

    path = os.path.join(
        save_dir,
        filename
    )

    model.load_state_dict(

        torch.load(
            path,
            map_location=device
        )
    )

    model.eval()

    return model

In [ ]:
def load_han_models(
    save_dir,
    data,
    device
):

    models = {}

    for filename in os.listdir(save_dir):

        if not filename.endswith(".pth"):
            continue

        if not filename.startswith("DBLP_HAN"):
            continue

        parts = filename.replace(
            ".pth",
            ""
        ).split("_")

        hidden_channels = int(
            parts[2][1:]
        )

        heads = int(
            parts[3].replace(
                "heads",
                ""
            )
        )

        dropout = float(
            parts[4][1:]
        )

        model = HAN(

            in_channels={

                node_type:
                data[node_type].num_features

                for node_type in data.node_types

                if 'x' in data[node_type]

            },

            metadata=data.metadata(),

            hidden_channels=hidden_channels,

            out_channels=4,

            heads=heads,

            dropout=dropout

        ).to(device)

        model.load_state_dict(

            torch.load(

                os.path.join(
                    save_dir,
                    filename
                ),

                map_location=device

            )

        )

        model.eval()

        models[
            filename
        ] = model

    return models

In [ ]:
@torch.no_grad()
def select_nodes_hetero(
    models,
    data,
    n_nodes=20,
    seed=42
):

    predictions = []

    for model in models.values():

        model.eval()

        out = model(
            data.x_dict,
            data.edge_index_dict
        )

        predictions.append(
            out.argmax(dim=1).cpu()
        )

    y_true = data['author'].y.cpu()

    test_mask = (
        data['author']
        .test_mask
        .cpu()
    )

    valid_nodes = []

    for node_id in test_mask.nonzero(
        as_tuple=True
    )[0]:

        node_id = node_id.item()

        preds = [

            pred[node_id].item()

            for pred in predictions
        ]

        # todos concordam?
        agreement = len(
            set(preds)
        ) == 1

        # todos acertam?
        correct = all(

            pred == y_true[node_id].item()

            for pred in preds
        )

        if agreement and correct:

            valid_nodes.append(
                node_id
            )

    print(
        f"Nós elegíveis: {len(valid_nodes)}"
    )

    torch.manual_seed(seed)

    perm = torch.randperm(
        len(valid_nodes)
    )

    selected_nodes = [

        valid_nodes[i]

        for i in perm[
            :min(
                n_nodes,
                len(valid_nodes)
            )
        ]
    ]

    return selected_nodes

In [ ]:
class HANIntegratedWrapper(torch.nn.Module):

    def __init__(self, model, data):
        super().__init__()

        self.model = model
        self.edge_index_dict = data.edge_index_dict

        self.paper_x = data["paper"].x
        self.term_x = data["term"].x

    def forward(self, author_x):

        x_dict = {
            "author": author_x,
            "paper": self.paper_x,
            "term": self.term_x
        }

        out = self.model(
            x_dict,
            self.edge_index_dict
        )

        return out

In [ ]:
def create_integrated_gradients_hetero(
    model,
    data
):

    wrapped_model = HANIntegratedWrapper(
        model,
        data
    )

    ig = IntegratedGradients(
        wrapped_model
    )

    return ig

In [ ]:
def get_top_features_hetero_ig(
    attribution,
    k=10
):

    top_features = (
        np.abs(attribution)
        .argsort()[-k:][::-1]
        .tolist()
    )

    return top_features

In [ ]:
def make_explanations_hetero_ig(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        ig = create_integrated_gradients_hetero(
            model,
            data
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            pred = model(
                data.x_dict,
                data.edge_index_dict
            ).argmax(dim=1)

        for node_id in tqdm(selected_nodes):

            target_class = pred[
                node_id
            ].item()

            attrs = ig.attribute(

                inputs=data["author"].x,

                target=target_class

            )

            node_attr = (
                attrs[node_id]
                .detach()
                .cpu()
                .numpy()
            )

            top_features = get_top_features_hetero_ig(

                node_attr,

                k=k
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [ ]:
def make_explanations_attention_features(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            _, embeddings = model(

                data.x_dict,

                data.edge_index_dict,

                return_embeddings=True
            )

        for node_id in selected_nodes:

            node_embedding = (
                embeddings[node_id]
                .cpu()
                .numpy()
            )

            top_features = (

                np.abs(node_embedding)
                .argsort()[-k:][::-1]
                .tolist()
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [ ]:
def create_saliency_hetero(
    model,
    data
):

    wrapped_model = HANIntegratedWrapper(
        model,
        data
    )

    saliency = Saliency(
        wrapped_model
    )

    return saliency

In [ ]:
def get_top_features_saliency(
    attribution,
    k=10
):

    top_features = (

        np.abs(
            attribution
        )

        .argsort()[-k:][::-1]

        .tolist()

    )

    return top_features

In [ ]:
def make_explanations_hetero_saliency(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        saliency = create_saliency_hetero(
            model,
            data
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            pred = model(

                data.x_dict,

                data.edge_index_dict

            ).argmax(dim=1)

        for node_id in tqdm(
            selected_nodes
        ):

            target_class = pred[
                node_id
            ].item()

            attrs = saliency.attribute(

                inputs=data["author"].x,

                target=target_class

            )

            node_attr = (

                attrs[node_id]

                .detach()

                .cpu()

                .numpy()

            )

            top_features = get_top_features_saliency(

                node_attr,

                k=k

            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [ ]:


def save_explanations(
    explanations,
    dataset,
    model,
    explainer,
    save_dir
):

    save_dir = Path(save_dir)

    save_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        f"{dataset}_"
        f"{model}_"
        f"{explainer}.pkl"
    )

    filepath = save_dir / filename

    with open(
        filepath,
        "wb"
    ) as f:

        pickle.dump(
            explanations,
            f
        )

    print(
        f"Salvo em: {filepath}"
    )

In [ ]:


def load_explanations(
    dataset,
    model,
    explainer,
    save_dir
):

    filepath = (

        Path(save_dir)

        /

        f"{dataset}_{model}_{explainer}.pkl"
    )

    with open(
        filepath,
        "rb"
    ) as f:

        return pickle.load(f)

In [ ]:
def stability(
    explanations,
    selected_nodes
):

    model_ids = list(
        explanations.keys()
    )

    if len(model_ids) < 2:

        raise ValueError(
            "São necessários pelo menos 2 modelos."
        )

    stability_results = []

    for node_id in selected_nodes:

        jaccards = []
        overlaps = []
        spearmans = []

        for model_a, model_b in combinations(
            model_ids,
            2
        ):

            features_a = explanations[
                model_a
            ][node_id]

            features_b = explanations[
                model_b
            ][node_id]

            score_jaccard = (
                jaccard_similarity(
                    features_a,
                    features_b
                )
            )

            score_overlap = (
                overlap_at_k(
                    features_a,
                    features_b
                )
            )

            score_spearman = (
                spearman_topk(
                    features_a,
                    features_b
                )
            )

            if np.isnan(
                score_spearman
            ):
                score_spearman = 0

            jaccards.append(
                score_jaccard
            )

            overlaps.append(
                score_overlap
            )

            spearmans.append(
                score_spearman
            )

        stability_results.append({

            "node": node_id,

            "mean_jaccard":
                np.mean(jaccards),

            "mean_overlap":
                np.mean(overlaps),

            "mean_spearman":
                np.mean(spearmans),

            "std_jaccard":
                np.std(jaccards),

            "std_overlap":
                np.std(overlaps),

            "std_spearman":
                np.std(spearmans)

        })

    return pd.DataFrame(
        stability_results
    )

## Metricas

In [44]:
def jaccard_similarity(a, b):

    a = set(a)
    b = set(b)

    return len(a & b) / len(a | b)

In [45]:
def overlap_at_k(a, b):

    return len(
        set(a) & set(b)
    )

In [46]:


def spearman_topk(a, b):

    common = list(
        set(a) & set(b)
    )

    if len(common) < 2:
        return 0

    rank_a = [
        a.index(x)
        for x in common
    ]

    rank_b = [
        b.index(x)
        for x in common
    ]

    corr, _ = spearmanr(
        rank_a,
        rank_b
    )

    return corr

## seleção de nodes

In [47]:
SEED=42

In [48]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)

In [49]:
selected_nodes_dblp = select_nodes_hetero(

    han_models,

    dblp_new,

    n_nodes=20

)

print(
    selected_nodes_dblp
)

Nós elegíveis: 2375
[278, 3644, 2798, 1326, 1489, 2171, 1229, 3237, 3342, 323, 1750, 2483, 908, 4043, 2019, 3020, 3124, 202, 1696, 3928]


## Integrated Gradient

In [74]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [ ]:
explanations_han_ig = make_explanations_hetero_ig(

    han_models,

    selected_nodes_dblp,

    dblp_new
)

Explicando DBLP_HAN_h128_heads2_d0.7.pth


100%|██████████| 20/20 [01:25<00:00,  4.29s/it]


Explicando DBLP_HAN_h128_heads8_d0.7.pth


100%|██████████| 20/20 [01:25<00:00,  4.26s/it]


Explicando DBLP_HAN_h64_heads8_d0.7.pth


100%|██████████| 20/20 [01:14<00:00,  3.72s/it]


In [70]:
stability_han_ig_df = stability(

    explanations_han_ig,

    selected_nodes_dblp

)
stability_han_ig_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,278,1.000000,10.000000,0.713131,0.000000,0.000000,0.070215
1,3644,1.000000,10.000000,0.830303,0.000000,0.000000,0.095443
2,2798,0.818182,9.000000,0.116667,0.000000,0.000000,0.506806
3,1326,0.337302,5.000000,0.338095,0.072955,0.816497,0.044160
4,1489,0.465201,6.333333,0.209524,0.051803,0.471405,0.358632


In [71]:
print(
    stability_han_ig_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.745045
mean_overlap     8.216667
mean_spearman    0.537897
dtype: float64


In [72]:
save_explanations(
    explanations=explanations_han_ig,
    dataset="DBLP",
    model="HAN",
    explainer="IntegratedGradients",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\DBLP_HAN_IntegratedGradients.pkl


## Saliency

In [88]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [89]:
explanations_han_saliency = make_explanations_hetero_saliency(

    han_models,

    selected_nodes_dblp,

    dblp_new

)

Explicando DBLP_HAN_h128_heads2_d0.7.pth


100%|██████████| 20/20 [00:09<00:00,  2.16it/s]


Explicando DBLP_HAN_h128_heads8_d0.7.pth


100%|██████████| 20/20 [00:12<00:00,  1.61it/s]


Explicando DBLP_HAN_h64_heads8_d0.7.pth


100%|██████████| 20/20 [00:07<00:00,  2.80it/s]


In [90]:
stability_han_saliency_df = stability(

    explanations_han_saliency,

    selected_nodes_dblp

)
stability_han_saliency_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,278,0.017544,0.333333,0.000000,0.024811,0.471405,0.000000
1,3644,0.037037,0.666667,-0.333333,0.052378,0.942809,0.471405
2,2798,0.093911,1.666667,0.333333,0.058378,0.942809,0.471405
3,1326,0.072125,1.333333,0.333333,0.027568,0.471405,0.471405
4,1489,0.017544,0.333333,0.000000,0.024811,0.471405,0.000000


In [91]:
print(
    stability_han_saliency_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.238899
mean_overlap     2.700000
mean_spearman    0.185000
dtype: float64


In [92]:
save_explanations(
    explanations=explanations_han_saliency,
    dataset="DBLP",
    model="HAN",
    explainer="Saliency",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\DBLP_HAN_Saliency.pkl


## Attention-based explanation

In [80]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [81]:
explanations_han_af = make_explanations_attention_features(

    han_models,

    selected_nodes_dblp,

    dblp_new
)

Explicando DBLP_HAN_h128_heads2_d0.7.pth
Explicando DBLP_HAN_h128_heads8_d0.7.pth
Explicando DBLP_HAN_h64_heads8_d0.7.pth


In [82]:
stability_han_af_df = stability(explanations_han_af, selected_nodes_dblp)
stability_han_af_df.head()

,node,mean_jaccard,mean_overlap,mean_spearman,std_jaccard,std_overlap,std_spearman
0,278,0.054581,1.000000,0.333333,0.045382,0.816497,0.471405
1,3644,0.017544,0.333333,0.000000,0.024811,0.471405,0.000000
2,2798,0.058824,1.000000,0.166667,0.083189,1.414214,0.235702
3,1326,0.058824,1.000000,-0.166667,0.083189,1.414214,0.235702
4,1489,0.074074,1.333333,0.000000,0.052378,0.942809,0.816497


In [83]:
print(
    stability_han_af_df[
        [
            "mean_jaccard",
            "mean_overlap",
            "mean_spearman"
        ]
    ].mean()
)

mean_jaccard     0.052425
mean_overlap     0.950000
mean_spearman    0.033333
dtype: float64


In [84]:
save_explanations(
    explanations=explanations_han_af,
    dataset="DBLP",
    model="HAN",
    explainer="AttentionFeatures",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

Salvo em: C:\faculdade\Tabalho-Final-XAI\explicacoes\DBLP_HAN_AttentionFeatures.pkl


## Resultados

In [ ]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_resultsV2.pkl",
    "rb"
) as f:

    all_results_gnnexplainer = pickle.load(f)

In [ ]:
import pandas as pd

summary_rows = []

for experiment_name, df in all_results_gnnexplainer.items():

    dataset, model = experiment_name.split("_")

    summary_rows.append({

        "Dataset": dataset,

        "Modelo": model,

        "Explicador": "GNNExplainer",

        "Jaccard":
            df["mean_jaccard"].mean(),

        "Overlap@10":
            df["mean_overlap"].mean(),

        "Spearman":
            df["mean_spearman"].mean(),

        "Jaccard Std":
            df["mean_jaccard"].std()

    })

summary_df = pd.DataFrame(
    summary_rows
)

summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
0,Cora,GCN,GNNExplainer,0.703752,8.150000,0.233770,0.148330
1,Cora,GAT,GNNExplainer,0.579527,7.100000,0.129131,0.205590
2,Cora,GraphSAGE,GNNExplainer,0.679780,7.900000,0.271140,0.202846
3,CiteSeer,GCN,GNNExplainer,0.407731,5.483333,-0.137619,0.204636
4,CiteSeer,GAT,GNNExplainer,0.341761,4.983333,-0.014881,0.082806
5,CiteSeer,GraphSAGE,GNNExplainer,0.452192,6.066667,0.037976,0.142583
6,PubMed,GCN,GNNExplainer,0.265057,4.016667,-0.063968,0.118881
7,PubMed,GAT,GNNExplainer,0.228724,3.516667,-0.078532,0.132584
8,PubMed,GraphSAGE,GNNExplainer,0.282567,4.133333,-0.008690,0.160023


In [ ]:
all_results_graphlime = {

    "Cora_GCN":
        stability_graphlime_cora_GCN_df_conf,

    "Cora_GAT":
        stability_graphlime_cora_GAT_df_conf,

    "Cora_GraphSAGE":
        stability_graphlime_cora_GraphSAGE_df_conf,

    "CiteSeer_GCN":
        stability_graphlime_citeseer_GCN_df_conf,

    "CiteSeer_GAT":
        stability_graphlime_citeseer_GAT_df_conf,

    "CiteSeer_GraphSAGE":   
        stability_graphlime_citeseer_GraphSAGE_df_conf,

    "PubMed_GCN":
        stability_graphlime_pubmed_GCN_df_conf,
        
    "PubMed_GAT":
        stability_graphlime_pubmed_GAT_df_conf,
    "PubMed_GraphSAGE":
        stability_graphlime_pubmed_GraphSAGE_df_conf
}

In [ ]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results_graphlimeV2.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results_graphlime,
        f
    )

In [ ]:
summary_rows = []

for explainer_name, results in {

    "GNNExplainer":
        all_results_gnnexplainer,

    "GraphLIME":
        all_results_graphlime

}.items():

    for experiment_name, df in results.items():

        dataset, model = experiment_name.split("_")

        summary_rows.append({

            "Dataset": dataset,

            "Modelo": model,

            "Explicador": explainer_name,

            "Jaccard":
                df["mean_jaccard"].mean(),

            "Overlap@10":
                df["mean_overlap"].mean(),

            "Spearman":
                df["mean_spearman"].mean(),

            "Jaccard Std":
                df["mean_jaccard"].std()

        })

estabilidade_summary_df = pd.DataFrame(
    summary_rows
)

estabilidade_summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
0,Cora,GCN,GNNExplainer,0.703752,8.150000,0.233770,0.148330
1,Cora,GAT,GNNExplainer,0.579527,7.100000,0.129131,0.205590
2,Cora,GraphSAGE,GNNExplainer,0.679780,7.900000,0.271140,0.202846
3,CiteSeer,GCN,GNNExplainer,0.407731,5.483333,-0.137619,0.204636
4,CiteSeer,GAT,GNNExplainer,0.341761,4.983333,-0.014881,0.082806
5,CiteSeer,GraphSAGE,GNNExplainer,0.452192,6.066667,0.037976,0.142583
6,PubMed,GCN,GNNExplainer,0.265057,4.016667,-0.063968,0.118881
7,PubMed,GAT,GNNExplainer,0.228724,3.516667,-0.078532,0.132584
8,PubMed,GraphSAGE,GNNExplainer,0.282567,4.133333,-0.008690,0.160023
9,Cora,GCN,GraphLIME,0.523854,6.283333,0.633095,0.312560


In [ ]:
estabilidade_summary_df = estabilidade_summary_df.sort_values(

    [
        "Dataset",
        "Modelo",
        "Explicador"
    ]
)

estabilidade_summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
4,CiteSeer,GAT,GNNExplainer,0.341761,4.983333,-0.014881,0.082806
13,CiteSeer,GAT,GraphLIME,0.694488,7.566667,0.710245,0.310995
3,CiteSeer,GCN,GNNExplainer,0.407731,5.483333,-0.137619,0.204636
12,CiteSeer,GCN,GraphLIME,0.655719,7.366667,0.605657,0.321787
5,CiteSeer,GraphSAGE,GNNExplainer,0.452192,6.066667,0.037976,0.142583
14,CiteSeer,GraphSAGE,GraphLIME,0.561899,6.466667,0.573918,0.352038
1,Cora,GAT,GNNExplainer,0.579527,7.100000,0.129131,0.205590
10,Cora,GAT,GraphLIME,0.735250,7.983333,0.786613,0.320134
0,Cora,GCN,GNNExplainer,0.703752,8.150000,0.233770,0.148330
9,Cora,GCN,GraphLIME,0.523854,6.283333,0.633095,0.312560


In [ ]:
estabilidade_summary_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\estabilidade_dfV2.csv",
    index=False
)

In [ ]:
all_results_integrated_gradients = {

    "Cora_GCN":
        stability_integrated_gradients_cora_GCN_df,

    "Cora_GAT":
        stability_integrated_gradients_cora_GAT_df,

    "Cora_GraphSAGE":
        stability_integrated_gradients_cora_GraphSAGE_df,

    "CiteSeer_GCN":
        stability_integrated_gradients_citeseer_GCN_df,

    "CiteSeer_GAT":
        stability_integrated_gradients_citeseer_GAT_df,

    "CiteSeer_GraphSAGE":   
        stability_integrated_gradients_citeseer_GraphSAGE_df,

    "PubMed_GCN":
        stability_integrated_gradients_pubmed_GCN_df,
        
    "PubMed_GAT":
        stability_integrated_gradients_pubmed_GAT_df,
    "PubMed_GraphSAGE":
        stability_integrated_gradients_pubmed_GraphSAGE_df
}

In [ ]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\all_stability_results_integrated_gradients.pkl",
    "wb"
) as f:

    pickle.dump(
        all_results_integrated_gradients,
        f
    )

In [ ]:
summary_rows = []

for explainer_name, results in {

    "GNNExplainer":
        all_results_gnnexplainer,

    "GraphLIME":
        all_results_graphlime,
    
    "IntegratedGradients":
        all_results_integrated_gradients

}.items():

    for experiment_name, df in results.items():

        dataset, model = experiment_name.split("_")

        summary_rows.append({

            "Dataset": dataset,

            "Modelo": model,

            "Explicador": explainer_name,

            "Jaccard":
                df["mean_jaccard"].mean(),

            "Overlap@10":
                df["mean_overlap"].mean(),

            "Spearman":
                df["mean_spearman"].mean(),

            "Jaccard Std":
                df["mean_jaccard"].std()

        })

estabilidade_summary_df = pd.DataFrame(
    summary_rows
)

estabilidade_summary_df

,Dataset,Modelo,Explicador,Jaccard,Overlap@10,Spearman,Jaccard Std
0,Cora,GCN,GNNExplainer,0.703752,8.150000,0.233770,0.148330
1,Cora,GAT,GNNExplainer,0.579527,7.100000,0.129131,0.205590
2,Cora,GraphSAGE,GNNExplainer,0.679780,7.900000,0.271140,0.202846
3,CiteSeer,GCN,GNNExplainer,0.407731,5.483333,-0.137619,0.204636
4,CiteSeer,GAT,GNNExplainer,0.341761,4.983333,-0.014881,0.082806
5,CiteSeer,GraphSAGE,GNNExplainer,0.452192,6.066667,0.037976,0.142583
6,PubMed,GCN,GNNExplainer,0.265057,4.016667,-0.063968,0.118881
7,PubMed,GAT,GNNExplainer,0.228724,3.516667,-0.078532,0.132584
8,PubMed,GraphSAGE,GNNExplainer,0.282567,4.133333,-0.008690,0.160023
9,Cora,GCN,GraphLIME,0.523854,6.283333,0.633095,0.312560


In [ ]:
estabilidade_summary_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\estabilidade_dfV2.csv",
    index=False
)

# Fidelidade

In [5]:
TOP_DIR = "C:\\faculdade\\Tabalho-Final-XAI\\top_models"

## Dados

In [45]:
DBdataset = DBLP(
    root='data/DBLP'
)

dblp = DBdataset[0]


dblp_new = HeteroData()

# nós
dblp_new['author'].x = dblp['author'].x
dblp_new['author'].y = dblp['author'].y
dblp_new['author'].train_mask = dblp['author'].train_mask
dblp_new['author'].val_mask = dblp['author'].val_mask
dblp_new['author'].test_mask = dblp['author'].test_mask

dblp_new['paper'].x = dblp['paper'].x

dblp_new['term'].x = dblp['term'].x

dblp_new['author', 'to', 'paper'].edge_index = \
    dblp['author', 'to', 'paper'].edge_index

dblp_new['paper', 'to', 'author'].edge_index = \
    dblp['paper', 'to', 'author'].edge_index

dblp_new['paper', 'to', 'term'].edge_index = \
    dblp['paper', 'to', 'term'].edge_index

dblp_new['term', 'to', 'paper'].edge_index = \
    dblp['term', 'to', 'paper'].edge_index

## Apoio

In [49]:


warnings.filterwarnings(
    "ignore",
    category=UserWarning
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

warnings.filterwarnings(
    "ignore"
)

In [50]:
def load_han_model(
    hidden_channels,
    heads,
    dropout,
    data,
    device,
    save_dir
):

    model = HAN(

        in_channels={

            node_type:
            data[node_type].num_features

            for node_type in data.node_types

            if 'x' in data[node_type]

        },

        metadata=data.metadata(),

        hidden_channels=hidden_channels,

        out_channels=4,

        heads=heads,

        dropout=dropout

    ).to(device)

    filename = (

        f"DBLP_HAN"

        f"_h{hidden_channels}"

        f"_heads{heads}"

        f"_d{dropout}.pth"

    )

    path = os.path.join(
        save_dir,
        filename
    )

    model.load_state_dict(

        torch.load(
            path,
            map_location=device
        )
    )

    model.eval()

    return model

In [51]:
def load_han_models(
    save_dir,
    data,
    device
):

    models = {}

    for filename in os.listdir(save_dir):

        if not filename.endswith(".pth"):
            continue

        if not filename.startswith("DBLP_HAN"):
            continue

        parts = filename.replace(
            ".pth",
            ""
        ).split("_")

        hidden_channels = int(
            parts[2][1:]
        )

        heads = int(
            parts[3].replace(
                "heads",
                ""
            )
        )

        dropout = float(
            parts[4][1:]
        )

        model = HAN(

            in_channels={

                node_type:
                data[node_type].num_features

                for node_type in data.node_types

                if 'x' in data[node_type]

            },

            metadata=data.metadata(),

            hidden_channels=hidden_channels,

            out_channels=4,

            heads=heads,

            dropout=dropout

        ).to(device)

        model.load_state_dict(

            torch.load(

                os.path.join(
                    save_dir,
                    filename
                ),

                map_location=device

            )

        )

        model.eval()

        models[
            filename
        ] = model

    return models

In [52]:
@torch.no_grad()
def select_nodes_hetero(
    models,
    data,
    n_nodes=20,
    seed=42
):

    predictions = []

    for model in models.values():

        model.eval()

        out = model(
            data.x_dict,
            data.edge_index_dict
        )

        predictions.append(
            out.argmax(dim=1).cpu()
        )

    y_true = data['author'].y.cpu()

    test_mask = (
        data['author']
        .test_mask
        .cpu()
    )

    valid_nodes = []

    for node_id in test_mask.nonzero(
        as_tuple=True
    )[0]:

        node_id = node_id.item()

        preds = [

            pred[node_id].item()

            for pred in predictions
        ]

        # todos concordam?
        agreement = len(
            set(preds)
        ) == 1

        # todos acertam?
        correct = all(

            pred == y_true[node_id].item()

            for pred in preds
        )

        if agreement and correct:

            valid_nodes.append(
                node_id
            )

    print(
        f"Nós elegíveis: {len(valid_nodes)}"
    )

    torch.manual_seed(seed)

    perm = torch.randperm(
        len(valid_nodes)
    )

    selected_nodes = [

        valid_nodes[i]

        for i in perm[
            :min(
                n_nodes,
                len(valid_nodes)
            )
        ]
    ]

    return selected_nodes

In [53]:
class HANIntegratedWrapper(torch.nn.Module):

    def __init__(self, model, data):
        super().__init__()

        self.model = model
        self.edge_index_dict = data.edge_index_dict

        self.paper_x = data["paper"].x
        self.term_x = data["term"].x

    def forward(self, author_x):

        x_dict = {
            "author": author_x,
            "paper": self.paper_x,
            "term": self.term_x
        }

        out = self.model(
            x_dict,
            self.edge_index_dict
        )

        return out

In [54]:
def create_integrated_gradients_hetero(
    model,
    data
):

    wrapped_model = HANIntegratedWrapper(
        model,
        data
    )

    ig = IntegratedGradients(
        wrapped_model
    )

    return ig

In [55]:
def get_top_features_hetero_ig(
    attribution,
    k=10
):

    top_features = (
        np.abs(attribution)
        .argsort()[-k:][::-1]
        .tolist()
    )

    return top_features

In [56]:
def make_explanations_hetero_ig(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        ig = create_integrated_gradients_hetero(
            model,
            data
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            pred = model(
                data.x_dict,
                data.edge_index_dict
            ).argmax(dim=1)

        for node_id in tqdm(selected_nodes):

            target_class = pred[
                node_id
            ].item()

            attrs = ig.attribute(

                inputs=data["author"].x,

                target=target_class

            )

            node_attr = (
                attrs[node_id]
                .detach()
                .cpu()
                .numpy()
            )

            top_features = get_top_features_hetero_ig(

                node_attr,

                k=k
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [57]:
def make_explanations_attention_features(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            _, embeddings = model(

                data.x_dict,

                data.edge_index_dict,

                return_embeddings=True
            )

        for node_id in selected_nodes:

            node_embedding = (
                embeddings[node_id]
                .cpu()
                .numpy()
            )

            top_features = (

                np.abs(node_embedding)
                .argsort()[-k:][::-1]
                .tolist()
            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [ ]:
def create_saliency_hetero(
    model,
    data
):

    wrapped_model = HANIntegratedWrapper(
        model,
        data
    )

    saliency = Saliency(
        wrapped_model
    )

    return saliency

In [59]:
def get_top_features_saliency(
    attribution,
    k=10
):

    top_features = (

        np.abs(
            attribution
        )

        .argsort()[-k:][::-1]

        .tolist()

    )

    return top_features

In [60]:
def make_explanations_hetero_saliency(
    models,
    selected_nodes,
    data,
    k=10
):

    explanations = {}

    for model_name, model in models.items():

        print(
            f"Explicando {model_name}"
        )

        saliency = create_saliency_hetero(
            model,
            data
        )

        explanations[
            model_name
        ] = {}

        with torch.no_grad():

            pred = model(

                data.x_dict,

                data.edge_index_dict

            ).argmax(dim=1)

        for node_id in tqdm(
            selected_nodes
        ):

            target_class = pred[
                node_id
            ].item()

            attrs = saliency.attribute(

                inputs=data["author"].x,

                target=target_class

            )

            node_attr = (

                attrs[node_id]

                .detach()

                .cpu()

                .numpy()

            )

            top_features = get_top_features_saliency(

                node_attr,

                k=k

            )

            explanations[
                model_name
            ][node_id] = top_features

    return explanations

In [61]:


def save_explanations(
    explanations,
    dataset,
    model,
    explainer,
    save_dir
):

    save_dir = Path(save_dir)

    save_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    filename = (
        f"{dataset}_"
        f"{model}_"
        f"{explainer}.pkl"
    )

    filepath = save_dir / filename

    with open(
        filepath,
        "wb"
    ) as f:

        pickle.dump(
            explanations,
            f
        )

    print(
        f"Salvo em: {filepath}"
    )

In [ ]:
def load_explanations(
    dataset,
    model,
    explainer,
    save_dir
):

    filepath = (

        Path(save_dir)

        /

        f"{dataset}_{model}_{explainer}.pkl"
    )

    with open(
        filepath,
        "rb"
    ) as f:

        return pickle.load(f)

In [63]:
def fidelity_score_hetero(
    model,
    data,
    node_id,
    important_features
):

    model.eval()

    with torch.no_grad():

        original_out = model(
            data.x_dict,
            data.edge_index_dict
        )

        original_prob = torch.softmax(
            original_out[node_id],
            dim=0
        )

        pred_class = original_prob.argmax().item()

        original_confidence = original_prob[
            pred_class
        ].item()

    # clone do x_dict
    masked_x_dict = {

        key: value.clone()

        for key, value in data.x_dict.items()
    }

    # mascara apenas features do author
    masked_x_dict["author"][
        node_id,
        important_features
    ] = 0

    with torch.no_grad():

        masked_out = model(
            masked_x_dict,
            data.edge_index_dict
        )

        masked_prob = torch.softmax(
            masked_out[node_id],
            dim=0
        )

        masked_confidence = masked_prob[
            pred_class
        ].item()

    fidelity = (
        original_confidence
        -
        masked_confidence
    )

    return fidelity

In [64]:
def fidelity_hetero(
    models,
    explanations,
    selected_nodes,
    data
):

    fidelity_results = []

    for node_id in selected_nodes:

        scores = []

        for model_name, model in models.items():

            important_features = explanations[
                model_name
            ][node_id]

            score = fidelity_score_hetero(

                model,

                data,

                node_id,

                important_features

            )

            scores.append(
                score
            )

        fidelity_results.append({

            "node": node_id,

            "mean_fidelity":
                np.mean(scores),

            "std_fidelity":
                np.std(scores),

            "min_fidelity":
                np.min(scores),

            "max_fidelity":
                np.max(scores)

        })

    return pd.DataFrame(
        fidelity_results
    )

## Metricas

In [46]:
def jaccard_similarity(a, b):

    a = set(a)
    b = set(b)

    return len(a & b) / len(a | b)

In [47]:
def overlap_at_k(a, b):

    return len(
        set(a) & set(b)
    )

In [48]:


def spearman_topk(a, b):

    common = list(
        set(a) & set(b)
    )

    if len(common) < 2:
        return 0

    rank_a = [
        a.index(x)
        for x in common
    ]

    rank_b = [
        b.index(x)
        for x in common
    ]

    corr, _ = spearmanr(
        rank_a,
        rank_b
    )

    return corr

## seleção de nodes

In [72]:
SEED=42

In [73]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)

In [74]:
selected_nodes_dblp = select_nodes_hetero(

    han_models,

    dblp_new,

    n_nodes=20

)

print(
    selected_nodes_dblp
)

Nós elegíveis: 2375
[278, 3644, 2798, 1326, 1489, 2171, 1229, 3237, 3342, 323, 1750, 2483, 908, 4043, 2019, 3020, 3124, 202, 1696, 3928]


## Integrated Gradient

In [95]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [96]:
dblp_han_ig_exp = load_explanations(
    dataset="DBLP",
    model="HAN",
    explainer="IntegratedGradients",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [97]:
fidelity_han_ig = fidelity_hetero(

    models=han_models,

    explanations=dblp_han_ig_exp,

    selected_nodes=selected_nodes_dblp,

    data=dblp_new
)
fidelity_han_ig.head()

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,-0.000016,0.000020,-4.434586e-05,0.000000
1,3644,0.001069,0.000970,0.000000e+00,0.002348
2,2798,0.000018,0.000013,0.000000e+00,0.000028
3,1326,0.000827,0.000936,1.192093e-07,0.002136
4,1489,0.000012,0.000015,0.000000e+00,0.000033


In [98]:
fidelity_han_ig

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,-1.583497e-05,2.020145e-05,-4.434586e-05,0.000000
1,3644,1.069446e-03,9.699983e-04,0.000000e+00,0.002348
2,2798,1.849731e-05,1.309154e-05,0.000000e+00,0.000028
3,1326,8.265376e-04,9.363239e-04,1.192093e-07,0.002136
4,1489,1.170238e-05,1.505788e-05,0.000000e+00,0.000033
5,2171,6.004771e-04,6.311527e-04,2.384186e-07,0.001473
6,1229,-2.807379e-05,5.042701e-05,-9.888411e-05,0.000015
7,3237,2.380212e-05,1.686466e-05,0.000000e+00,0.000037
8,3342,-3.973643e-08,5.619580e-08,-1.192093e-07,0.000000
9,323,-4.410744e-06,8.388105e-06,-1.615286e-05,0.000003


## Saliency

In [79]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [81]:
dblp_han_sc_exp = load_explanations(
    dataset="DBLP",
    model="HAN",
    explainer="Saliency",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [82]:
fidelity_han_sc = fidelity_hetero(

    models=han_models,

    explanations=dblp_han_sc_exp,

    selected_nodes=selected_nodes_dblp,

    data=dblp_new
)
fidelity_han_sc.head()

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,0.000000,0.000000,0.000000e+00,0.000000
1,3644,0.000195,0.000275,0.000000e+00,0.000584
2,2798,0.000008,0.000008,0.000000e+00,0.000018
3,1326,0.000595,0.000689,1.192093e-07,0.001561
4,1489,0.000007,0.000008,0.000000e+00,0.000018


In [83]:
fidelity_han_sc

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
1,3644,1.948078e-04,2.754999e-04,0.000000e+00,0.000584
2,2798,7.867813e-06,7.554051e-06,0.000000e+00,0.000018
3,1326,5.952517e-04,6.888479e-04,1.192093e-07,0.001561
4,1489,6.556511e-06,8.200929e-06,0.000000e+00,0.000018
5,2171,2.373457e-04,3.356575e-04,0.000000e+00,0.000712
6,1229,-1.025200e-05,1.449852e-05,-3.075600e-05,0.000000
7,3237,5.205472e-06,7.361650e-06,0.000000e+00,0.000016
8,3342,-7.947286e-08,1.123916e-07,-2.384186e-07,0.000000
9,323,0.000000e+00,0.000000e+00,0.000000e+00,0.000000


## Attention-based explanation

In [84]:
han_models = load_han_models(

    TOP_DIR,

    dblp_new,

    device

)


In [85]:
dblp_han_af_exp = load_explanations(
    dataset="DBLP",
    model="HAN",
    explainer="AttentionFeatures",
    save_dir=r"C:\faculdade\Tabalho-Final-XAI\explicacoes"
)

In [86]:
fidelity_han_af = fidelity_hetero(

    models=han_models,

    explanations=dblp_han_af_exp,

    selected_nodes=selected_nodes_dblp,

    data=dblp_new
)
fidelity_han_af.head()

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,0.000000e+00,0.000000e+00,0.000000,0.000000
1,3644,0.000000e+00,0.000000e+00,0.000000,0.000000
2,2798,0.000000e+00,0.000000e+00,0.000000,0.000000
3,1326,-4.144510e-05,5.861222e-05,-0.000124,0.000000
4,1489,6.357829e-07,8.991328e-07,0.000000,0.000002


In [87]:
fidelity_han_af

,node,mean_fidelity,std_fidelity,min_fidelity,max_fidelity
0,278,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
1,3644,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
2,2798,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
3,1326,-4.144510e-05,5.861222e-05,-0.000124,0.000000e+00
4,1489,6.357829e-07,8.991328e-07,0.000000,1.907349e-06
5,2171,2.533197e-05,3.582482e-05,0.000000,7.599592e-05
6,1229,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
7,3237,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
8,3342,0.000000e+00,0.000000e+00,0.000000,0.000000e+00
9,323,0.000000e+00,0.000000e+00,0.000000,0.000000e+00


## Resultados

In [99]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_han_ig.pkl",
    "wb"
) as f:

    pickle.dump(
        fidelity_han_ig,
        f
    )

In [89]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_han_sc.pkl",
    "wb"
) as f:

    pickle.dump(
        fidelity_han_sc,
        f
    )

In [90]:
import pickle

with open(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_han_af.pkl",
    "wb"
) as f:

    pickle.dump(
        fidelity_han_af,
        f
    )

In [102]:
fidelity_han_ig.columns

Index(['node', 'mean_fidelity', 'std_fidelity', 'min_fidelity',
       'max_fidelity'],
      dtype='str')

In [104]:
summary_rows = []

for explainer_name, df in {

    "IntegratedGradients":
        fidelity_han_ig,

    "Saliency":
        fidelity_han_sc,

    "AttentionFeatures":
        fidelity_han_af

}.items():

    summary_rows.append({

        "Dataset": "DBLP",

        "Explicador": explainer_name,

        "Fidelity":
            df["mean_fidelity"].mean(),

        "Fidelity Std":
            df["mean_fidelity"].std(),

        "Min Fidelity":
            df["min_fidelity"].mean(),

        "Max Fidelity":
            df["max_fidelity"].mean()

    })

fidelity_summary_df = pd.DataFrame(
    summary_rows
)

fidelity_summary_df = fidelity_summary_df.sort_values(
    ["Dataset", "Explicador"]
)

fidelity_summary_df

,Dataset,Explicador,Fidelity,Fidelity Std,Min Fidelity,Max Fidelity
2,DBLP,AttentionFeatures,-0.000007,0.000021,-0.000025,0.000004
0,DBLP,IntegratedGradients,0.000126,0.000320,-0.000036,0.000331
1,DBLP,Saliency,0.000047,0.000147,-0.000021,0.000150


In [105]:
fidelity_summary_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_global_heterogenio.csv",
    index=False
)

In [108]:
summary_rows = []

for explainer_name, df in {

    "IntegratedGradients":
        fidelity_han_ig,

    "Saliency":
        fidelity_han_sc,

    "AttentionFeatures":
        fidelity_han_af

}.items():

    for _, row in df.iterrows():

        summary_rows.append({

            "Dataset": "DBLP",

            "Modelo": "HAN",

            "Explicador": explainer_name,

            "Node": row["node"],

            "Mean Fidelity":
                row["mean_fidelity"],

            "Std Fidelity":
                row["std_fidelity"],

            "Min Fidelity":
                row["min_fidelity"],

            "Max Fidelity":
                row["max_fidelity"]

        })

fidelity_node_df = pd.DataFrame(
    summary_rows
)

fidelity_node_df

,Dataset,Modelo,Explicador,Node,Mean Fidelity,Std Fidelity,Min Fidelity,Max Fidelity
0,DBLP,HAN,IntegratedGradients,278.0,-1.583497e-05,2.020145e-05,-4.434586e-05,0.000000e+00
1,DBLP,HAN,IntegratedGradients,3644.0,1.069446e-03,9.699983e-04,0.000000e+00,2.348185e-03
2,DBLP,HAN,IntegratedGradients,2798.0,1.849731e-05,1.309154e-05,0.000000e+00,2.843142e-05
3,DBLP,HAN,IntegratedGradients,1326.0,8.265376e-04,9.363239e-04,1.192093e-07,2.135754e-03
4,DBLP,HAN,IntegratedGradients,1489.0,1.170238e-05,1.505788e-05,0.000000e+00,3.296137e-05
5,DBLP,HAN,IntegratedGradients,2171.0,6.004771e-04,6.311527e-04,2.384186e-07,1.472712e-03
6,DBLP,HAN,IntegratedGradients,1229.0,-2.807379e-05,5.042701e-05,-9.888411e-05,1.466274e-05
7,DBLP,HAN,IntegratedGradients,3237.0,2.380212e-05,1.686466e-05,0.000000e+00,3.701448e-05
8,DBLP,HAN,IntegratedGradients,3342.0,-3.973643e-08,5.619580e-08,-1.192093e-07,0.000000e+00
9,DBLP,HAN,IntegratedGradients,323.0,-4.410744e-06,8.388105e-06,-1.615286e-05,2.920628e-06


In [109]:
fidelity_node_df.to_csv(
    r"C:\faculdade\Tabalho-Final-XAI\results\fidelity_node_heterogenio.csv",
    index=False
)